In [1]:
%load_ext nb_black
import os
from glob import glob
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import pickle
from scipy import stats
from os.path import join
import scipy
from pathlib import Path
from collections import defaultdict
from datetime import datetime

<IPython.core.display.Javascript object>

In [2]:
plt.rcParams.update(
    {
        "font.size": 6,  # Default font size for all text
        "axes.titlesize": 6,  # Axes title
        "axes.labelsize": 6,  # Axes labels
        "xtick.labelsize": 6,  # X-axis tick labels
        "ytick.labelsize": 6,  # Y-axis tick labels
        "legend.fontsize": 6,  # Legend text
        "figure.titlesize": 6,  # Figure title
        "lines.linewidth": 0.5,  # Default line width
        "figure.figsize": (9 / 2.54, 6 / 2.54),  # Default figure size in cm
        "figure.dpi": 150,  # Default DPI for figures
        "xtick.major.size": 1,  # X-axis major tick length
        "ytick.major.size": 1,  # Y-axis major tick length
        "figure.constrained_layout.use": True,  # Use constrained layout by default
    }
)

<IPython.core.display.Javascript object>

In [3]:
def get_meta_from_file_path(file_path):
    meta = file_path.split("/")
    version = meta[-2]
    name = meta[-1].split("_")
    var = name[0]
    mdl = name[2]
    exp = name[3]
    label = name[4]
    time = name[-1].split(".")[0].split("-")
    time0 = time[0]
    time1 = time[1]

    return var, mdl, exp, label, version, time0, time1

<IPython.core.display.Javascript object>

In [4]:
cmip6_data_dir = Path.cwd().parent
res_dir = Path.cwd().parent / "processed"

<IPython.core.display.Javascript object>

In [5]:
# get all the variable file paths, get rid of the EC-Earth3-Veg becasue it is handled seperately
file_paths = sorted(
    [
        i
        for i in cmip6_data_dir.rglob("*.nc")
        if "sftlf" not in i.name
        and "areacella" not in i.name
        and "processed" not in str(i)
    ]
)

print(len(file_paths))

20569


<IPython.core.display.Javascript object>

In [6]:
mdl_exp_label = []
mdl_exp_label_version = []
to_remove_file_paths = []

for f in file_paths:
    var, mdl, exp, label, version, time0, time1 = get_meta_from_file_path(str(f))

    if label in ["r1i1p1f1", "r1i1p1f2", "r4i1p1f1"] and int(time0) <= 210012:
        mdl_exp_label.append(mdl + "_" + exp + "_" + label)
        mdl_exp_label_version.append(mdl + "_" + exp + "_" + label + "_" + version)
    else:
        to_remove_file_paths.append(f)

mdl_exp_label = sorted(list(set(mdl_exp_label)))
mdl_exp_label_version = sorted(list(set(mdl_exp_label_version)))
to_remove_file_paths = sorted(to_remove_file_paths)

file_paths = [i for i in file_paths if i not in to_remove_file_paths]
len(file_paths)

20569

<IPython.core.display.Javascript object>

In [7]:
# from zipfile import ZipFile, ZIP_DEFLATED

# zip_path = Path("/Users/gatang/Research/Data/CMIP6/data_after_2100.zip")
# with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
#     for f in to_remove_file_paths:
#         zf.write(f, arcname=f.name)

<IPython.core.display.Javascript object>

In [8]:
# for f in to_remove_file_paths:
#     f.unlink()

<IPython.core.display.Javascript object>

In [9]:
mdl_exp_label

['ACCESS-ESM1-5_1pctCO2-bgc_r1i1p1f1',
 'ACCESS-ESM1-5_1pctCO2-rad_r1i1p1f1',
 'ACCESS-ESM1-5_1pctCO2_r1i1p1f1',
 'ACCESS-ESM1-5_historical_r1i1p1f1',
 'ACCESS-ESM1-5_ssp126_r1i1p1f1',
 'ACCESS-ESM1-5_ssp245_r1i1p1f1',
 'ACCESS-ESM1-5_ssp370_r1i1p1f1',
 'ACCESS-ESM1-5_ssp585_r1i1p1f1',
 'BCC-CSM2-MR_1pctCO2-bgc_r1i1p1f1',
 'BCC-CSM2-MR_1pctCO2_r1i1p1f1',
 'BCC-CSM2-MR_historical_r1i1p1f1',
 'BCC-CSM2-MR_ssp126_r1i1p1f1',
 'BCC-CSM2-MR_ssp245_r1i1p1f1',
 'BCC-CSM2-MR_ssp370_r1i1p1f1',
 'BCC-CSM2-MR_ssp585_r1i1p1f1',
 'CESM2-WACCM_1pctCO2_r1i1p1f1',
 'CESM2-WACCM_historical_r1i1p1f1',
 'CESM2-WACCM_ssp126_r1i1p1f1',
 'CESM2-WACCM_ssp245_r1i1p1f1',
 'CESM2-WACCM_ssp370_r1i1p1f1',
 'CESM2-WACCM_ssp585_r1i1p1f1',
 'CESM2_1pctCO2-bgc_r1i1p1f1',
 'CESM2_1pctCO2-rad_r1i1p1f1',
 'CESM2_1pctCO2_r1i1p1f1',
 'CESM2_historical_r1i1p1f1',
 'CMCC-CM2-SR5_1pctCO2_r1i1p1f1',
 'CMCC-CM2-SR5_historical_r1i1p1f1',
 'CMCC-CM2-SR5_ssp126_r1i1p1f1',
 'CMCC-CM2-SR5_ssp245_r1i1p1f1',
 'CMCC-CM2-SR5_ssp370_r1i1

<IPython.core.display.Javascript object>

In [10]:
mdl_exp_label_version

['ACCESS-ESM1-5_1pctCO2-bgc_r1i1p1f1_v20191118',
 'ACCESS-ESM1-5_1pctCO2-rad_r1i1p1f1_v20191118',
 'ACCESS-ESM1-5_1pctCO2_r1i1p1f1_v20191115',
 'ACCESS-ESM1-5_historical_r1i1p1f1_v20191115',
 'ACCESS-ESM1-5_ssp126_r1i1p1f1_v20210318',
 'ACCESS-ESM1-5_ssp245_r1i1p1f1_v20191115',
 'ACCESS-ESM1-5_ssp370_r1i1p1f1_v20191115',
 'ACCESS-ESM1-5_ssp585_r1i1p1f1_v20210318',
 'BCC-CSM2-MR_1pctCO2-bgc_r1i1p1f1_v20190313',
 'BCC-CSM2-MR_1pctCO2-bgc_r1i1p1f1_v20190321',
 'BCC-CSM2-MR_1pctCO2_r1i1p1f1_v20181012',
 'BCC-CSM2-MR_1pctCO2_r1i1p1f1_v20181015',
 'BCC-CSM2-MR_historical_r1i1p1f1_v20181114',
 'BCC-CSM2-MR_historical_r1i1p1f1_v20181126',
 'BCC-CSM2-MR_ssp126_r1i1p1f1_v20190314',
 'BCC-CSM2-MR_ssp245_r1i1p1f1_v20190308',
 'BCC-CSM2-MR_ssp245_r1i1p1f1_v20190314',
 'BCC-CSM2-MR_ssp370_r1i1p1f1_v20190308',
 'BCC-CSM2-MR_ssp370_r1i1p1f1_v20190314',
 'BCC-CSM2-MR_ssp585_r1i1p1f1_v20190308',
 'BCC-CSM2-MR_ssp585_r1i1p1f1_v20190314',
 'CESM2-WACCM_1pctCO2_r1i1p1f1_v20190425',
 'CESM2-WACCM_historical

<IPython.core.display.Javascript object>

In [11]:
import warnings
from xarray.coding.times import SerializationWarning

warnings.filterwarnings("ignore", category=SerializationWarning)

<IPython.core.display.Javascript object>

In [12]:
def get_combined_dataset(identifier):
    print("start:", identifier)

    file_sel = sorted([f for f in file_paths if identifier in f.name])

    files_by_var = defaultdict(list)
    for f in file_sel:
        var = f.stem.split("_")[0]
        files_by_var[var].append(f)

    # if multiple versions for a single variable exist, use the latest version
    for var, files in files_by_var.items():
        versions = [f.parent.name for f in files]
        latest_version = max(versions)
        files_by_var[var] = [f for f in files if f.parent.name == latest_version]

    da_dict = {}
    ds_attrs = None
    for var, files in files_by_var.items():
        temp_ds = xr.open_mfdataset(
            sorted(files),
            combine="by_coords",
            data_vars="minimal",
            coords="minimal",
            compat="override",
        )
        if ds_attrs is None:
            ds_attrs = temp_ds.attrs.copy()
        da = temp_ds[var].load()
        if "height" in da.coords:
            da = da.reset_coords("height", drop=True)
        if "type" in da.coords:
            da = da.reset_coords("type", drop=True)
        da_dict[var] = da
        temp_ds.close()

    try:
        ds = xr.merge(da_dict.values(), compat="override", join="exact")
    except ValueError:
        ref_var = next(iter(da_dict))
        ref_da = da_dict[ref_var]

        same_spatial_coords = all(
            da["lat"].equals(ref_da["lat"]) and da["lon"].equals(ref_da["lon"])
            for da in da_dict.values()
        )

        if not same_spatial_coords:
            same_spatial_shape = all(
                da.sizes.get("lat") == ref_da.sizes.get("lat")
                and da.sizes.get("lon") == ref_da.sizes.get("lon")
                for da in da_dict.values()
            )

            if same_spatial_shape:
                print(
                    f"lat/lon differ across variables but have the same size, "
                    f"reindex to {ref_var} coordinates using nearest neighbour"
                )
                for var, da in da_dict.items():
                    da = da.reindex(lat=ref_da.lat, lon=ref_da.lon, method="nearest")
                    assert da.lat.equals(ref_da.lat) and da.lon.equals(
                        ref_da.lon
                    ), f"Spatial coordinates do not match for {var}"
                    da_dict[var] = da
            else:
                raise ValueError("lat/lon dimensions have different sizes \n")

        try:
            ds = xr.merge(da_dict.values(), compat="override", join="exact")
        except ValueError:
            time_start = min(da.time.values[0] for da in da_dict.values())
            time_end = max(da.time.values[-1] for da in da_dict.values())
            print(
                f"time differs across variables, use the total time period "
                f"{time_start} to {time_end}"
            )
            ds = xr.merge(da_dict.values(), compat="override", join="outer")

    ds.attrs = ds_attrs

    return ds

<IPython.core.display.Javascript object>

In [13]:
def get_annual_dataset(ds):
    days_in_month = ds.time.dt.days_in_month
    days_in_year = days_in_month.groupby("time.year").sum()

    annual_data = {}
    for var in ds.data_vars:
        da = ds[var]
        units = da.attrs.get("units")

        if units == "K" or units == "kg m-2":
            annual = (da * days_in_month).groupby("time.year").sum(
                min_count=1
            ) / days_in_year
            annual = annual.astype(da.dtype)
            annual.attrs = da.attrs.copy()

        elif units == "kg m-2 s-1":
            annual = (da * days_in_month * 86400).groupby("time.year").sum(min_count=1)
            annual = annual.astype(da.dtype)
            annual.attrs = da.attrs.copy()
            annual.attrs["units"] = "kg m-2 yr-1"

        else:
            print(f"Skipping {var}: units = {da.attrs.get('units')}")
            continue

        annual_data[var] = annual

    ds_annual = xr.Dataset(annual_data)
    ds_annual = ds_annual.rename({"year": "time"})

    # some model outputs for idealized experiments show 101-250 (ACCESS-ESM1-5) or 1-150 ()
    # shift them to start from 1850
    if ds_annual.time[0].data < 1850:
        original_start = ds_annual.time[0].data
        original_end = ds_annual.time[-1].data
        shift = 1850 - original_start

        ds_annual["time"] = ds_annual.time + shift

        print(
            f"Time range {original_start}-{original_end}, "
            f"shifted to 1850-{ds_annual.time[-1].data}"
        )

    ds_annual.attrs = {
        "experiment": ds.attrs.get("experiment"),
        "variant_label": ds.attrs.get("variant_label"),
        "source": ds.attrs.get("source"),
        "source_id": ds.attrs.get("source_id"),
        "summary": "Annual data derived from original CMIP6 monthly data.",
        "creator_and_contact": "Gang Tang; gang.tang.au@gmail.com",
        "creation_date": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        "comment": (
            "Original CMIP6 monthly data were processed to annual data. "
            "State variables were aggregated as annual means (e.g., annual mean cLand on grids), "
            "while flux variables were aggregated as annual totals (e.g., annual total npp on grids). "
            "The month-to-annual aggregation was weighted by the number of days in each month, "
            "using the original calendar provided in the monthly data. "
            "For idealized experiments, some models reported years 001-150 or 101-250, "
            "while most models used 1850-2000. "
            "We shifted model years beginning with 001 or 101 to 1850 to make them consistant."
        ),
    }

    return ds_annual

<IPython.core.display.Javascript object>

In [14]:
%%time

for identifier in mdl_exp_label:
    output_path = res_dir / "annual_data"/ f"{identifier}_annual_processed_by_Gang_Tang.nc"

    if output_path.exists():
        print(f"Already exists, skip: {output_path.name}")
        continue
    
    ds = get_combined_dataset(identifier)
    ds_annual = get_annual_dataset(ds)
    print()

    encoding = {
        var: {"zlib": True, "complevel": 4, "shuffle": True}
        for var in ds_annual.data_vars
    }
    ds_annual.to_netcdf(output_path, encoding=encoding)
    
    ds.close()
    ds_annual.close()

start: ACCESS-ESM1-5_1pctCO2-bgc_r1i1p1f1
Time range 101-250, shifted to 1850-1999

start: ACCESS-ESM1-5_1pctCO2-rad_r1i1p1f1
Time range 101-250, shifted to 1850-1999

start: ACCESS-ESM1-5_1pctCO2_r1i1p1f1
Time range 101-250, shifted to 1850-1999

start: ACCESS-ESM1-5_historical_r1i1p1f1

start: ACCESS-ESM1-5_ssp126_r1i1p1f1

start: ACCESS-ESM1-5_ssp245_r1i1p1f1

start: ACCESS-ESM1-5_ssp370_r1i1p1f1

start: ACCESS-ESM1-5_ssp585_r1i1p1f1

start: BCC-CSM2-MR_1pctCO2-bgc_r1i1p1f1
lat/lon differ across variables but have the same size, reindex to tas coordinates using nearest neighbour
time differs across variables, use the total time period 1850-01-16 12:00:00 to 2000-12-16 12:00:00

start: BCC-CSM2-MR_1pctCO2_r1i1p1f1
lat/lon differ across variables but have the same size, reindex to tas coordinates using nearest neighbour
time differs across variables, use the total time period 1850-01-16 12:00:00 to 2000-12-16 12:00:00

start: BCC-CSM2-MR_historical_r1i1p1f1
lat/lon differ across varia


start: EC-Earth3-Veg_ssp370_r4i1p1f1

start: EC-Earth3-Veg_ssp434_r1i1p1f1
lat/lon differ across variables but have the same size, reindex to cLitter coordinates using nearest neighbour

start: EC-Earth3-Veg_ssp460_r1i1p1f1
lat/lon differ across variables but have the same size, reindex to cLitter coordinates using nearest neighbour

start: EC-Earth3-Veg_ssp585_r1i1p1f1
lat/lon differ across variables but have the same size, reindex to cLitter coordinates using nearest neighbour

start: EC-Earth3-Veg_ssp585_r4i1p1f1

start: IPSL-CM6A-LR_1pctCO2-bgc_r1i1p1f1

start: IPSL-CM6A-LR_1pctCO2_r1i1p1f1

start: IPSL-CM6A-LR_historical_r1i1p1f1

start: IPSL-CM6A-LR_ssp119_r1i1p1f1

start: IPSL-CM6A-LR_ssp126_r1i1p1f1

start: IPSL-CM6A-LR_ssp245_r1i1p1f1

start: IPSL-CM6A-LR_ssp370_r1i1p1f1

start: IPSL-CM6A-LR_ssp460_r1i1p1f1

start: IPSL-CM6A-LR_ssp585_r1i1p1f1

start: MIROC-ES2L_1pctCO2-bgc_r1i1p1f2

start: MIROC-ES2L_1pctCO2-rad_r1i1p1f2

start: MIROC-ES2L_1pctCO2Ndep-bgc_r1i1p1f2

start: MI

<IPython.core.display.Javascript object>